# Accumulated neuron interactions

Teacher width 8; student widths 8/16; 200k steps; ten paired seeds. Both layers train. Momentum (lr 0.1, beta 0.9) and SGD (lr 1). Read `docs/cumulative_interference.md` for the protocol and interpretation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/neural_competition").exists())
results = root / "runs/pier_study/trajectory/analysis"
effects = pd.read_csv(results / "paired_effects.csv")

def figure(name):
    display(Image(filename=str(results / f"{name}.png")))

def summary(optimizer, view="reference"):
    selected = effects[(effects.optimizer == optimizer) & (effects.view == view) & (effects.metric == "S_plus")].copy()
    for key in ["width8", "width16", "difference", "low", "high"]:
        selected[key] *= 100
    return selected[["comparison", "width8", "width16", "difference", "low", "high", "paired_seeds"]]


## Quantities and comparison clocks

$A_r$ is the standalone half-MSE change, and $J_{rs}=L_{rs}-L_r-L_s+L_0$.
Positive $J$ is an unfavorable interaction. Define
$G=\sum_r[-A_r]_+$, $H=\sum_r[A_r]_+$,
$C_+=\sum_{r<s}[J_{rs}]_+$ and $C_-=\sum_{r<s}[-J_{rs}]_+$.
Then $\Delta L=H-G+C_+-C_-$ exactly for each update.

We plot cumulative sums and $S_+^{cum}=\sum C_+/\sum G$; compare at the same
$U=\sum G$; and pool ratios within bins of the same fixed-reference MSE.
The scalar matched-loss comparison weights 12 log-spaced bins from 0.2 to
0.02 equally. All ten paired seeds qualify. Other lower-loss bins can have
fewer paired seeds. Every update is measured; no checkpoint interpolation
is used to accumulate the totals.

In [ ]:
figure("comparison_summary")

## Momentum: primary comparisons

Values below are percentages of summed standalone helpful gains. Differences and intervals are percentage points. The reference is one fixed 512-example training subset. The source view uses each update's own batch; its changing objectives do not telescope.

In [ ]:
display(summary("momentum"))
display(summary("momentum", "source"))

In [ ]:
figure("momentum_cumulative")
figure("momentum_source_cumulative")

In [ ]:
figure("momentum_equal_gains")

Equal U is equal accumulated standalone helpful activity, not equal net progress. Only the U range reached by every run is shown; no seed drops out of these curves.

In [ ]:
figure("momentum_matched_loss")

The rightmost panels show coverage. The successful narrow seeds form a selected subset at low losses; curves there do not describe all narrow runs. The advantage can reverse in individual later bins.

In [ ]:
figure("momentum_local_windows")

## SGD control

The control uses lr 1 and no momentum. Do not interpret cross-optimizer penalty differences independently of their step sizes.

In [ ]:
display(summary("sgd"))
figure("sgd_cumulative")
figure("sgd_source_cumulative")
figure("sgd_equal_gains")

In [ ]:
figure("sgd_matched_loss")
figure("sgd_local_windows")

## Fitting and numerical checks

In [ ]:
display(pd.read_csv(results / "final_fit_summary.csv"))
display(pd.read_csv(results / "audit.csv").groupby("view")[["max_local_error", "telescope_error", "reference_adjacency_error"]].max())
print(json.loads((results / "independent_audit.json").read_text()))

## Reuse the data

The following loads scalar diagnostics lazily; it does not load all of the roughly 0.9 GB file. Axis order is updates, models, view, field. The config contains model identities and field names. All teacher/data and raw-run files remain under the ignored `runs/` directory.

In [ ]:
folder = results.parent / "momentum"
config = json.loads((folder / "config.json").read_text())
updates = np.load(folder / "updates.npy", mmap_mode="r")
print("Shape:", updates.shape)
print("Views:", config["views"])
print("Fields:", config["fields"])
display(pd.DataFrame(config["models"]))


## Interpretation

The controls leave a smaller width advantage than the raw cumulative totals suggest. These are exact local finite-step interactions, including ordinary diminishing returns. The measurements do not show that the penalty is avoidable or that eliminating it would cause the final fitting advantage. Learning-rate, neuron decomposition, training phase and survivor selection still matter.